Train a variational autoencoder on the image dataset of your choice, and
use it to generate images. Alternatively, you can try to find an unlabeled
dataset that you are interested in and see if you can generate new
samples.

In [ ]:
# Si estás en Colab, NO reinstales tensorflow salvo que sea necesario.
# %pip install -q -U tensorflow

import tensorflow as tf
from tensorflow.keras import mixed_precision

tf.random.set_seed(42)

print("TF:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

# 1) Evitar que TF reserve toda la VRAM al inicio
gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

# 2) Mixed precision (muy recomendado en RTX modernas)
mixed_precision.set_global_policy("float32")

# 3) (Opcional) XLA JIT: puede acelerar algunos modelos
tf.config.optimizer.set_jit(True)

# 4) Estrategia: solo usar MirroredStrategy si hay >1 GPU
strategy = tf.distribute.MirroredStrategy() if len(gpus) > 1 else tf.distribute.get_strategy()
print("Strategy:", type(strategy).__name__)
print("Policy:", mixed_precision.global_policy())


TF: 2.19.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Strategy: _DefaultDistributionStrategy
Policy: <DTypePolicy "float32">


In [2]:
%pip install tensorflow_datasets

In [3]:
import tensorflow_datasets as tdfs

class Sampling(tf.keras.Layer):
    def call(self, inputs):
        mean, log_var = inputs
        # Clip log_var para evitar exp(log_var/2) → inf y luego NaNs en la loss
        log_var = tf.clip_by_value(tf.cast(log_var, tf.float32), -20.0, 2.0)
        return tf.random.normal(tf.shape(log_var)) * tf.exp(log_var / 2) + mean

train_dataset, test_dataset = tdfs.load(
    "celeb_a",
    split=["train", "test"],
    with_info=False
)

def prep(ex):
    img = tf.cast(ex["image"], tf.float32) / 255.0
    return img, img   # <- input y target

train_ds = train_dataset.map(prep, num_parallel_calls=tf.data.AUTOTUNE).batch(128).prefetch(tf.data.AUTOTUNE)
test_ds  = test_dataset.map(prep,  num_parallel_calls=tf.data.AUTOTUNE).batch(128).prefetch(tf.data.AUTOTUNE)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/3 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/celeb_a/incomplete.P5SKOW_2.1.0/celeb_a-train.tfrecord*...:   0%|         …

Generating validation examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/celeb_a/incomplete.P5SKOW_2.1.0/celeb_a-validation.tfrecord*...:   0%|    …

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/celeb_a/incomplete.P5SKOW_2.1.0/celeb_a-test.tfrecord*...:   0%|          …

Dataset celeb_a downloaded and prepared to /root/tensorflow_datasets/celeb_a/2.1.0. Subsequent calls will reuse this data.


In [4]:
class KLDivergenceLayer(tf.keras.layers.Layer):
    def __init__(self, kl_weight=1.0, **kwargs):
        super().__init__(**kwargs)
        self.kl_weight = kl_weight

    def call(self, inputs):
        z_mean, z_log_var = inputs

        z_mean32 = tf.cast(z_mean, tf.float32)
        z_log_var32 = tf.clip_by_value(tf.cast(z_log_var, tf.float32), -10.0, 10.0)

        kl_per_sample = -0.5 * tf.reduce_sum(
            1 + z_log_var32 - tf.exp(z_log_var32) - tf.square(z_mean32),
            axis=-1
        )
        self.add_loss(self.kl_weight * tf.reduce_mean(kl_per_sample))

        return [z_mean, z_log_var] 

In [5]:
for batch in train_dataset.take(1):
    image = batch  # tensor (batch_size, 218, 178, 3)
    print(image)

{'attributes': {'5_o_Clock_Shadow': <tf.Tensor: shape=(), dtype=bool, numpy=False>, 'Arched_Eyebrows': <tf.Tensor: shape=(), dtype=bool, numpy=False>, 'Attractive': <tf.Tensor: shape=(), dtype=bool, numpy=True>, 'Bags_Under_Eyes': <tf.Tensor: shape=(), dtype=bool, numpy=False>, 'Bald': <tf.Tensor: shape=(), dtype=bool, numpy=False>, 'Bangs': <tf.Tensor: shape=(), dtype=bool, numpy=False>, 'Big_Lips': <tf.Tensor: shape=(), dtype=bool, numpy=True>, 'Big_Nose': <tf.Tensor: shape=(), dtype=bool, numpy=False>, 'Black_Hair': <tf.Tensor: shape=(), dtype=bool, numpy=False>, 'Blond_Hair': <tf.Tensor: shape=(), dtype=bool, numpy=False>, 'Blurry': <tf.Tensor: shape=(), dtype=bool, numpy=False>, 'Brown_Hair': <tf.Tensor: shape=(), dtype=bool, numpy=False>, 'Bushy_Eyebrows': <tf.Tensor: shape=(), dtype=bool, numpy=False>, 'Chubby': <tf.Tensor: shape=(), dtype=bool, numpy=False>, 'Double_Chin': <tf.Tensor: shape=(), dtype=bool, numpy=False>, 'Eyeglasses': <tf.Tensor: shape=(), dtype=bool, numpy=Fals

In [23]:
codings_size = 128

inputs = tf.keras.Input(shape=[218, 178, 3])
Z = tf.keras.layers.Flatten()(inputs)
Z = tf.keras.layers.Dense(600, activation="relu")(Z)
Z = tf.keras.layers.Dense(300, activation="relu")(Z)
Z = tf.keras.layers.Dense(150, activation="relu")(Z)
Z = tf.keras.layers.Dense(100, activation="relu")(Z)



codings_mean = tf.keras.layers.Dense(codings_size)(Z)
codings_log_variance = tf.keras.layers.Dense(codings_size)(Z)

codings_mean, codings_log_variance = KLDivergenceLayer(
    kl_weight=1.0 / (218 * 178 * 3)
)([codings_mean, codings_log_variance])

codings = Sampling()([codings_mean, codings_log_variance])

variational_encoder = tf.keras.Model(
    inputs=[inputs], outputs=[codings_mean, codings_log_variance, codings]
)

decoder_inputs = tf.keras.Input(shape=[codings_size])
x = tf.keras.layers.Dense(100, activation="relu")(decoder_inputs)
x = tf.keras.layers.Dense(150, activation="relu")(x)
x = tf.keras.layers.Dense(300, activation="relu")(x)
x = tf.keras.layers.Dense(600, activation="relu")(x)
x = tf.keras.layers.Dense(218 * 178 * 3)(x)
outputs = tf.keras.layers.Reshape([218, 178, 3])(x)

variational_decoder = tf.keras.Model(inputs=decoder_inputs, outputs=outputs)

In [24]:
_, _, codings = variational_encoder(inputs)
# Decoder con sigmoid para que salida esté en [0,1] y la MSE sea estable
reconstructions = variational_decoder(codings)
variational_autoencoder = tf.keras.Model(inputs=inputs, outputs=reconstructions)

In [25]:
variational_autoencoder.compile(loss="mse", optimizer=tf.keras.optimizers.Nadam())
variational_autoencoder.fit(train_ds, epochs=25, validation_data=test_ds)

Epoch 1/25
1272/1272 ━━━━━━━━━━━━━━━━━━━━ 126s 77ms/step - loss: 0.0473 - val_loss: 0.0273
Epoch 2/25
1272/1272 ━━━━━━━━━━━━━━━━━━━━ 53s 41ms/step - loss: 0.0270 - val_loss: 0.0245
Epoch 3/25
1272/1272 ━━━━━━━━━━━━━━━━━━━━ 53s 41ms/step - loss: 0.0252 - val_loss: 0.0245
Epoch 4/25
1272/1272 ━━━━━━━━━━━━━━━━━━━━ 53s 41ms/step - loss: 0.0240 - val_loss: 0.0230
Epoch 5/25
1272/1272 ━━━━━━━━━━━━━━━━━━━━ 53s 41ms/step - loss: 0.0232 - val_loss: 0.0221
Epoch 6/25
1272/1272 ━━━━━━━━━━━━━━━━━━━━ 53s 41ms/step - loss: 0.0223 - val_loss: 0.0219
Epoch 7/25
1272/1272 ━━━━━━━━━━━━━━━━━━━━ 53s 41ms/step - loss: 0.0215 - val_loss: 0.0207
Epoch 8/25
1272/1272 ━━━━━━━━━━━━━━━━━━━━ 53s 41ms/step - loss: 0.0209 - val_loss: 0.0207
Epoch 9/25
1272/1272 ━━━━━━━━━━━━━━━━━━━━ 53s 41ms/step - loss: 0.0204 - val_loss: 0.0211
Epoch 10/25
1272/1272 ━━━━━━━━━━━━━━━━━━━━ 53s 41ms/step - loss: 0.0200 - val_loss: 0.0198
Epoch 11/25
1272/1272 ━━━━━━━━━━━━━━━━━━━━ 53s 41ms/step - loss: 0.0197 - val_loss: 0.0190
Epoch 1

In [1]:
import matplotlib.pyplot as plt


# 1) sampleo latente
n = 1
z = tf.random.normal(shape=(n, codings_size))  # codings_size=128 en tu caso

# 2) genero imagen
gen = variational_decoder(z, training=False)
img = gen[0]  # primera imagen del batch

# 3) como tu decoder no tiene sigmoid final, clip a [0,1]
img = tf.clip_by_value(img, 0.0, 1.0)

# 4) mostrar
plt.figure(figsize=(4,4))
plt.imshow(img.numpy())
plt.axis("off")
plt.show()

NameError: name 'tf' is not defined